In [ ]:
# from google.colab import auth, drive
# auth.authenticate_user()
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/MIMIC_IV_v3.1/
# # Kéo toàn bộ dữ liệu từ Bucket về Drive siêu tốc
# !gsutil -m cp -r gs://mimic-ehr-v310/* /content/drive/MyDrive/MIMIC_IV_v3.1/


In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
import os, pathlib
os.chdir('/content/drive/MyDrive/MIMIC_IV_v3.1/hosp')

Mounted at /content/drive


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="mimic-ehr-v310")   # project ID thật của bạn
H = "physionet-data.mimiciv_3_1_hosp"

In [ ]:
!ls | head -10

admissions_000000000000.csv.gz
d_hcpcs_000000000000.csv.gz
diagnoses_icd_000000000000.csv.gz
d_icd_diagnoses_000000000000.csv.gz
d_icd_procedures_000000000000.csv.gz
d_labitems_000000000000.csv.gz
drgcodes_000000000000.csv.gz
emar_000000000000.csv.gz
emar_000000000001.csv.gz
emar_000000000002.csv.gz


In [ ]:
!ls -lh d_icd_diagnoses_000000000000.csv.gz

-rw------- 1 root root 0 Jun 30 07:40 d_icd_diagnoses_000000000000.csv.gz


**Các loại xét nghệm**

In [ ]:
import polars as pl

dlab_cols = ["itemid", "label", "fluid", "category"]
dlab = pl.read_csv(
    "d_labitems_000000000000.csv.gz",
    has_header=False,
    new_columns=dlab_cols,
)

ids = dlab.filter(pl.col("label").str.contains("(?i)hemoglobin|red blood"))
print(ids)

shape: (28, 4)
┌────────┬────────────────────────┬───────┬────────────┐
│ itemid ┆ label                  ┆ fluid ┆ category   │
│ ---    ┆ ---                    ┆ ---   ┆ ---        │
│ i64    ┆ str                    ┆ str   ┆ str        │
╞════════╪════════════════════════╪═══════╪════════════╡
│ 50805  ┆ Carboxyhemoglobin      ┆ Blood ┆ Blood Gas  │
│ 50811  ┆ Hemoglobin             ┆ Blood ┆ Blood Gas  │
│ 50814  ┆ Methemoglobin          ┆ Blood ┆ Blood Gas  │
│ 52032  ┆ P50 of Hemoglobin      ┆ Blood ┆ Blood Gas  │
│ 50852  ┆ % Hemoglobin A1c       ┆ Blood ┆ Chemistry  │
│ …      ┆ …                      ┆ …     ┆ …          │
│ 52128  ┆ Hemoglobin H Inclusion ┆ Blood ┆ Hematology │
│ 52129  ┆ Hemoglobin Other       ┆ Blood ┆ Hematology │
│ 52144  ┆ Methemoglobin          ┆ Blood ┆ Hematology │
│ 52157  ┆ Plasma Hemoglobin      ┆ Blood ┆ Hematology │
│ 52173  ┆ Red Blood Cell Enzyme  ┆ Blood ┆ Hematology │
└────────┴────────────────────────┴───────┴────────────┘


In [ ]:
import duckdb
con = duckdb.connect()

labevents_cols = {
    "labevent_id":"BIGINT","subject_id":"BIGINT","hadm_id":"BIGINT",
    "specimen_id":"BIGINT","itemid":"BIGINT","order_provider_id":"VARCHAR",
    "charttime":"TIMESTAMP","storetime":"TIMESTAMP","value":"VARCHAR",
    "valuenum":"DOUBLE","valueuom":"VARCHAR","ref_range_lower":"DOUBLE",
    "ref_range_upper":"DOUBLE","flag":"VARCHAR","priority":"VARCHAR","comments":"VARCHAR",
}
dlab_cols = {"itemid":"BIGINT","label":"VARCHAR","fluid":"VARCHAR","category":"VARCHAR"}

con.execute(f"""
CREATE OR REPLACE VIEW labevents AS
SELECT * FROM read_csv(
    'labevents_*.csv.gz',
    header=false,
    columns={labevents_cols},
    strict_mode=false,       -- chấp nhận dòng không chuẩn CSV
    ignore_errors=true,      -- bỏ qua dòng vẫn lỗi (nếu còn)
    escape='"'               -- hiểu "" là escape của dấu nháy
);

CREATE OR REPLACE VIEW d_labitems AS
SELECT * FROM read_csv('d_labitems_000000000000.csv.gz',
    header=false, columns={dlab_cols}, strict_mode=false, ignore_errors=true);
""")

In [ ]:
# 1. Đếm số label unique
con.execute("SELECT COUNT(DISTINCT label) AS n_unique_labels FROM d_labitems").df()

# 2. Liệt kê tất cả các label unique (kèm số itemid ứng với mỗi label)
con.execute("""
    SELECT label, COUNT(*) AS n_itemids
    FROM d_labitems
    GROUP BY label
    ORDER BY label
""").df()

,label,n_itemids
0,,1
1,Rbc,1
2,% Hemoglobin A1c,1
3,% Ionized Calcium,1
4,% Parasitemia,1
...,...,...
1187,reuse,1
1188,reuse1,1
1189,tacroFK,1
1190,wbcp,1


In [ ]:
# Join labevents với từ điển để biết itemid là xét nghiệm gì
df = con.execute("""
    SELECT l.subject_id, l.hadm_id, l.charttime,
           d.label, l.valuenum, l.valueuom
    FROM labevents l
    LEFT JOIN d_labitems d ON l.itemid = d.itemid
    WHERE d.label ILIKE '%hemoglobin%' OR d.label ILIKE '%red blood%'
    LIMIT 100
""").df()
df

,subject_id,hadm_id,charttime,label,valuenum,valueuom
0,13241244,21892522,2126-05-07 12:56:00,Hemoglobin,6.0,g/dL
1,14702642,25879036,2163-12-09 08:51:00,Hemoglobin,6.0,g/dL
2,11666451,22996335,2160-07-28 11:00:00,Hemoglobin,6.1,g/dL
3,12518169,22752177,2170-05-25 17:05:00,Hemoglobin,6.1,g/dL
4,15513126,29355614,2135-12-03 05:35:00,Hemoglobin,6.1,g/dL
...,...,...,...,...,...,...
95,15662316,22582725,2167-12-25 03:21:00,Hemoglobin,6.6,g/dL
96,17757913,27077024,2124-05-31 14:00:00,Hemoglobin,6.6,g/dL
97,18061354,26207405,2141-12-31 05:55:00,Hemoglobin,6.6,g/dL
98,15463295,24465523,2117-08-31 05:49:00,Hemoglobin,6.6,g/dL


**Time constraint**

In [ ]:
import duckdb
con = duckdb.connect()

adm_cols = {
    "subject_id":"BIGINT","hadm_id":"BIGINT","admittime":"TIMESTAMP","dischtime":"TIMESTAMP",
    "deathtime":"TIMESTAMP","admission_type":"VARCHAR","admit_provider_id":"VARCHAR",
    "admission_location":"VARCHAR","discharge_location":"VARCHAR","insurance":"VARCHAR",
    "language":"VARCHAR","marital_status":"VARCHAR","race":"VARCHAR",
    "edregtime":"TIMESTAMP","edouttime":"TIMESTAMP","hospital_expire_flag":"BIGINT",
}
con.execute(f"""
CREATE OR REPLACE VIEW admissions AS
SELECT * FROM read_csv('admissions_*.csv.gz', header=false, columns={adm_cols},
                       strict_mode=false, ignore_errors=true);
""")

# Phân bố số lần nhập viện
con.execute("""
    SELECT n_adm, COUNT(*) AS n_patients
    FROM (SELECT subject_id, COUNT(*) AS n_adm
          FROM admissions GROUP BY subject_id)
    GROUP BY n_adm ORDER BY n_adm
""").df()

,n_adm,n_patients
0,1,123289
1,2,44123
2,3,20108
3,4,11172
4,5,6629
...,...,...
92,138,1
93,142,1
94,163,1
95,185,1


In [ ]:
con.execute("""
    SELECT subject_id, hadm_id, admittime,
           EXTRACT(year FROM admittime) AS shifted_year
    FROM admissions
    WHERE subject_id IN (
        SELECT subject_id FROM admissions
        GROUP BY subject_id HAVING COUNT(DISTINCT EXTRACT(year FROM admittime)) > 1
        LIMIT 1
    )
    ORDER BY admittime
""").df()

,subject_id,hadm_id,admittime,shifted_year
0,16853337,20880689,2191-06-08 17:01:00,2191
1,16853337,29712263,2191-07-10 17:36:00,2191
2,16853337,25132910,2192-07-08 22:19:00,2192
3,16853337,23235036,2193-07-10 18:46:00,2193
4,16853337,28119398,2193-08-21 21:51:00,2193


In [ ]:
import duckdb
con = duckdb.connect()

labevents_cols = {
    "labevent_id":"BIGINT","subject_id":"BIGINT","hadm_id":"BIGINT","specimen_id":"BIGINT",
    "itemid":"BIGINT","order_provider_id":"VARCHAR","charttime":"TIMESTAMP","storetime":"TIMESTAMP",
    "value":"VARCHAR","valuenum":"DOUBLE","valueuom":"VARCHAR","ref_range_lower":"DOUBLE",
    "ref_range_upper":"DOUBLE","flag":"VARCHAR","priority":"VARCHAR","comments":"VARCHAR",
}
dlab_cols = {"itemid":"BIGINT","label":"VARCHAR","fluid":"VARCHAR","category":"VARCHAR"}
dx_cols   = {"subject_id":"BIGINT","hadm_id":"BIGINT","seq_num":"BIGINT","icd_code":"VARCHAR","icd_version":"BIGINT"}
dxd_cols  = {"icd_code":"VARCHAR","icd_version":"BIGINT","long_title":"VARCHAR"}

con.execute(f"""
CREATE OR REPLACE VIEW labevents AS SELECT * FROM read_csv('labevents_*.csv.gz',
    header=false, columns={labevents_cols}, strict_mode=false, ignore_errors=true, escape='"');
CREATE OR REPLACE VIEW d_labitems AS SELECT * FROM read_csv('d_labitems_000000000000.csv.gz',
    header=false, columns={dlab_cols}, strict_mode=false, ignore_errors=true);
CREATE OR REPLACE VIEW diagnoses_icd AS SELECT * FROM read_csv('diagnoses_icd_*.csv.gz',
    header=false, columns={dx_cols}, strict_mode=false, ignore_errors=true);
CREATE OR REPLACE VIEW d_icd_diagnoses AS SELECT * FROM read_csv('d_icd_diagnoses_*.csv.gz',
    header=false, columns={dxd_cols}, strict_mode=false, ignore_errors=true);
""")

In [ ]:
# # Thay đường dẫn bucket bằng của bạn. Lưu ý dấu "." ở cuối = tải về thư mục hiện tại
# !gsutil cp gs://mimic-ehr-v310/hosp/d_icd_diagnoses_000000000000.csv.gz /content/drive/MyDrive/MIMIC_IV_v3.1/hosp


Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mimic-ehr-v310/hosp/d_icd_diagnoses_000000000000.csv.gz...
- [1 files][851.4 KiB/851.4 KiB]                                                
Operation completed over 1 objects/851.4 KiB.                                    


**Xét nghiệm trên 1 bệnh nhân**

In [ ]:
PID = 16853337   # đổi thành subject_id bạn quan tâm

# (A) Bệnh nhân này được chẩn đoán bệnh gì, theo từng lần nằm viện
dx = con.execute(f"""
    SELECT d.subject_id, d.hadm_id, d.seq_num, d.icd_code, dd.long_title AS diagnosis
    FROM diagnoses_icd d
    LEFT JOIN d_icd_diagnoses dd
      ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
    WHERE d.subject_id = {PID}
    ORDER BY d.hadm_id, d.seq_num
""").df()

# (B) Bệnh nhân này được chỉ định xét nghiệm gì
lab = con.execute(f"""
    SELECT l.subject_id, l.hadm_id, l.charttime,
           li.label AS test_name, l.valuenum, l.valueuom, l.flag
    FROM labevents l
    LEFT JOIN d_labitems li ON l.itemid = li.itemid
    WHERE l.subject_id = {PID}
    ORDER BY l.hadm_id, l.charttime
""").df()

display(dx.head(50))
display(lab.head(50))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,subject_id,hadm_id,seq_num,icd_code,diagnosis
0,16853337,20880689,1,5789,"Hemorrhage of gastrointestinal tract, unspecified"
1,16853337,20880689,2,74739,Other anomalies of pulmonary artery and pulmon...
2,16853337,20880689,3,42731,Atrial fibrillation
3,16853337,20880689,4,4240,Mitral valve disorders
4,16853337,20880689,5,2809,"Iron deficiency anemia, unspecified"
5,16853337,20880689,6,2724,Other and unspecified hyperlipidemia
6,16853337,20880689,7,2352,"Neoplasm of uncertain behavior of stomach, int..."
7,16853337,20880689,8,V5861,Long-term (current) use of anticoagulants
8,16853337,20880689,9,41400,Coronary atherosclerosis of unspecified type o...
9,16853337,20880689,10,412,Old myocardial infarction


,subject_id,hadm_id,charttime,test_name,valuenum,valueuom,flag
0,16853337,20880689,2191-06-09 06:31:00,"Calcium, Total",8.50,mg/dL,None
1,16853337,20880689,2191-06-09 06:31:00,Chloride,105.00,mEq/L,None
2,16853337,20880689,2191-06-09 06:31:00,Glucose,88.00,mg/dL,None
3,16853337,20880689,2191-06-09 06:31:00,RDW,16.10,%,abnormal
4,16853337,20880689,2191-06-09 06:31:00,Magnesium,2.20,mg/dL,None
5,16853337,20880689,2191-06-09 06:31:00,PT,12.80,sec,abnormal
6,16853337,20880689,2191-06-09 06:31:00,Hematocrit,21.70,%,abnormal
7,16853337,20880689,2191-06-09 06:31:00,PTT,25.20,sec,None
8,16853337,20880689,2191-06-09 06:31:00,Anion Gap,11.00,mEq/L,None
9,16853337,20880689,2191-06-09 06:31:00,Red Blood Cells,2.43,m/uL,abnormal


In [ ]:
project_id = "mimic-ehr-v310"   # project GCP của bạn

from google.cloud import bigquery
client = bigquery.Client(project=project_id)

query = f"""
SELECT
    le.subject_id, le.hadm_id, le.charttime,
    dl.label AS test_name, le.valuenum, le.valueuom,
    dx.icd_code, dd.long_title AS diagnosis
FROM `physionet-data.mimiciv_3_1_hosp.labevents`        le
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_labitems`  dl USING (itemid)
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` dx
       ON le.subject_id = dx.subject_id AND le.hadm_id = dx.hadm_id
LEFT JOIN `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses` dd
       ON dx.icd_code = dd.icd_code AND dx.icd_version = dd.icd_version
WHERE le.subject_id = 16853337
ORDER BY le.hadm_id, le.charttime
"""
df = client.query(query).to_dataframe()
df.head(50)

,subject_id,hadm_id,charttime,test_name,valuenum,valueuom,icd_code,diagnosis
0,16853337,<NA>,2185-07-01 12:17:00,"Cholesterol, HDL",87.00,mg/dL,None,None
1,16853337,<NA>,2185-07-01 12:17:00,Folate,NaN,ng/mL,None,None
2,16853337,<NA>,2185-07-01 12:17:00,Monocytes,8.40,%,None,None
3,16853337,<NA>,2185-07-01 12:17:00,Ferritin,19.00,ng/mL,None,None
4,16853337,<NA>,2185-07-01 12:17:00,MCV,92.00,fL,None,None
5,16853337,<NA>,2185-07-01 12:17:00,White Blood Cells,6.20,K/uL,None,None
6,16853337,<NA>,2185-07-01 12:17:00,"Iron Binding Capacity, Total",373.00,ug/dL,None,None
7,16853337,<NA>,2185-07-01 12:17:00,Creatinine,0.80,mg/dL,None,None
8,16853337,<NA>,2185-07-01 12:17:00,Urea Nitrogen,20.00,mg/dL,None,None
9,16853337,<NA>,2185-07-01 12:17:00,Bicarbonate,28.00,mEq/L,None,None


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="mimic-ehr-v310")
H = "physionet-data.mimiciv_3_1_hosp"   # đổi đúng tên dataset nếu khác

# --- Bước 1: tìm 1 subject_id có chẩn đoán scoliosis ---
pid_df = client.query(f"""
  SELECT dx.subject_id, COUNT(*) AS n_dx
  FROM `{H}.diagnoses_icd` dx
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
  WHERE LOWER(dd.long_title) LIKE '%scoliosis%'
  GROUP BY dx.subject_id
  ORDER BY n_dx DESC
  LIMIT 1
""").to_dataframe()

PID = int(pid_df["subject_id"].iloc[0])
print("Chọn bệnh nhân:", PID)

Chọn bệnh nhân: 12934243


In [ ]:
result = client.query(f"""
  WITH dx_per_adm AS (
    SELECT subject_id, hadm_id,
           STRING_AGG(dd.long_title, ' | ' ORDER BY seq_num) AS diagnoses
    FROM `{H}.diagnoses_icd` dx
    LEFT JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
    GROUP BY subject_id, hadm_id
  )
  SELECT le.subject_id, le.hadm_id, le.charttime,
         dl.label AS test_name, le.valuenum, le.valueuom,
         x.diagnoses
  FROM `{H}.labevents` le
  LEFT JOIN `{H}.d_labitems` dl USING (itemid)
  LEFT JOIN dx_per_adm x USING (subject_id, hadm_id)
  WHERE le.subject_id = {PID}
  ORDER BY le.hadm_id, le.charttime
""").to_dataframe()
result.head(50)

,subject_id,hadm_id,charttime,test_name,valuenum,valueuom,diagnoses
0,12934243,<NA>,2140-11-01 20:00:00,Bicarbonate,29.000,mEq/L,None
1,12934243,<NA>,2140-11-01 20:00:00,MCH,31.500,pg,None
2,12934243,<NA>,2140-11-01 20:00:00,PTT,26.100,sec,None
3,12934243,<NA>,2140-11-01 20:00:00,PT,12.400,sec,None
4,12934243,<NA>,2140-11-01 20:00:00,Urea Nitrogen,14.000,mg/dL,None
5,12934243,<NA>,2140-11-01 20:00:00,Chloride,91.000,mEq/L,None
6,12934243,<NA>,2140-11-01 20:00:00,Sodium,126.000,mEq/L,None
7,12934243,<NA>,2140-11-01 20:00:00,Glucose,86.000,mg/dL,None
8,12934243,<NA>,2140-11-01 20:00:00,Lymphocytes,32.700,%,None
9,12934243,<NA>,2140-11-01 20:00:00,MCHC,33.900,%,None


In [ ]:
tb = client.query(f"""
  SELECT dx.subject_id, dx.hadm_id, dx.icd_code, dx.icd_version, dd.long_title AS diagnosis
  FROM `{H}.diagnoses_icd` dx
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
  WHERE LOWER(dd.long_title) LIKE '%tuberculosis%'
  ORDER BY dx.subject_id
""").to_dataframe()

print("Số dòng chẩn đoán lao:", len(tb))
print("Số bệnh nhân khác nhau:", tb["subject_id"].nunique())
tb.head(50)

Số dòng chẩn đoán lao: 3305
Số bệnh nhân khác nhau: 2020


,subject_id,hadm_id,icd_code,icd_version,diagnosis
0,10000560,28979390,V1201,9,Personal history of tuberculosis
1,10009049,22995465,5110,9,Pleurisy without mention of effusion or curren...
2,10024913,25022048,79551,9,Nonspecific reaction to tuberculin skin test w...
3,10024913,27207228,79551,9,Nonspecific reaction to tuberculin skin test w...
4,10024913,24005818,79551,9,Nonspecific reaction to tuberculin skin test w...
5,10025647,20807698,V1201,9,Personal history of tuberculosis
6,10025647,23201842,V1201,9,Personal history of tuberculosis
7,10028423,26381803,5110,9,Pleurisy without mention of effusion or curren...
8,10028423,26381803,79551,9,Nonspecific reaction to tuberculin skin test w...
9,10030549,21292378,Z8611,10,Personal history of tuberculosis


In [ ]:
tb_ids = client.query(f"""
  SELECT dx.subject_id, COUNT(*) AS n_tb_dx,
         STRING_AGG(DISTINCT dd.long_title, ' | ') AS tb_diagnoses
  FROM `{H}.diagnoses_icd` dx
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
  WHERE LOWER(dd.long_title) LIKE '%tuberculosis%'
  GROUP BY dx.subject_id
  ORDER BY n_tb_dx DESC
""").to_dataframe()
tb_ids

,subject_id,n_tb_dx,tb_diagnoses
0,18965447,27,Personal history of tuberculosis | Tuberculosi...
1,19061282,14,"Miliary tuberculosis, unspecified, unspecified..."
2,10332328,14,"Tuberculosis of vertebral column, unspecified ..."
3,13475033,13,Nonspecific reaction to tuberculin skin test w...
4,16676544,12,Latent tuberculosis | Nonspecific reaction to ...
...,...,...,...
2015,16332400,1,Sequelae of respiratory and unspecified tuberc...
2016,16937963,1,Personal history of tuberculosis
2017,17886844,1,Nonspecific reaction to tuberculin skin test w...
2018,18218042,1,Nonspecific reaction to tuberculin skin test w...


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="mimic-ehr-v310")   # project ID thật của bạn
I = "physionet-data.mimiciv_3_1_icu"                  # dataset icu (xác nhận tên trong console)

# Bước 1: tìm itemid của các vital sign cần
items = client.query(f"""
  SELECT itemid, label, unitname, category
  FROM `{I}.d_items`
  WHERE LOWER(label) LIKE '%heart rate%'
     OR LOWER(label) LIKE '%temperature%'
     OR LOWER(label) LIKE '%spo2%'
     OR LOWER(label) LIKE '%o2 saturation%'
  ORDER BY label
""").to_dataframe()
items

,itemid,label,unitname,category
0,226861,ART %O2 saturation (PA Line),%,PA Line Insertion
1,220227,Arterial O2 Saturation,%,Labs
2,226329,Blood Temperature CCO (C),°C,Routine Vital Signs
3,229236,Cerebral Temperature (C),°C,Hemodynamics
4,224674,Changes in Temperature,None,Toxicology
5,229862,Forehead SpO2 Sensor in Place,None,Routine Vital Signs
6,220045,Heart Rate,bpm,Routine Vital Signs
7,220047,Heart Rate Alarm - Low,bpm,Alarms
8,220046,Heart rate Alarm - High,bpm,Alarms
9,223769,O2 Saturation Pulseoxymetry Alarm - High,%,Alarms


In [ ]:
# Bước 2: join chartevents + d_items cho 1 bệnh nhân (PID từ phần scoliosis trước)
PID = 10332328	   # đổi thành subject_id bạn quan tâm

ce = client.query(f"""
  SELECT ce.subject_id, ce.hadm_id, ce.stay_id, ce.charttime,
         di.label AS measurement, ce.valuenum, ce.valueuom
  FROM `{I}.chartevents` ce
  LEFT JOIN `{I}.d_items` di USING (itemid)
  WHERE ce.subject_id = {PID}
    AND di.label IN ('Heart Rate', 'Temperature Celsius', 'O2 saturation pulseoxymetry')
  ORDER BY ce.stay_id, ce.charttime
""").to_dataframe()
ce.head(50)

,subject_id,hadm_id,stay_id,charttime,measurement,valuenum,valueuom
0,10332328,27237460,31344322,2198-03-09 17:41:00,O2 saturation pulseoxymetry,93.0,%
1,10332328,27237460,31344322,2198-03-09 17:41:00,Heart Rate,85.0,bpm
2,10332328,27237460,31344322,2198-03-09 18:00:00,Heart Rate,69.0,bpm
3,10332328,27237460,31344322,2198-03-09 18:00:00,O2 saturation pulseoxymetry,90.0,%
4,10332328,27237460,31344322,2198-03-09 19:00:00,Heart Rate,95.0,bpm
5,10332328,27237460,31344322,2198-03-09 19:17:00,O2 saturation pulseoxymetry,77.0,%
6,10332328,27237460,31344322,2198-03-09 20:00:00,O2 saturation pulseoxymetry,100.0,%
7,10332328,27237460,31344322,2198-03-09 20:00:00,Heart Rate,88.0,bpm
8,10332328,27237460,31344322,2198-03-09 21:00:00,O2 saturation pulseoxymetry,94.0,%
9,10332328,27237460,31344322,2198-03-09 21:00:00,Heart Rate,81.0,bpm


**Số lượng điểm dữ liệu/bệnh nhân trong 6 bệnh về đường hô hấp**

In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"

counts = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
)
SELECT
  COUNT(DISTINCT IF(title LIKE '%pneumonia%',                         subject_id, NULL)) AS pneumonia,
  COUNT(DISTINCT IF(title LIKE '%tuberculosis%',                      subject_id, NULL)) AS tuberculosis,
  COUNT(DISTINCT IF(title LIKE '%bronchitis%',                        subject_id, NULL)) AS bronchitis,
  COUNT(DISTINCT IF(title LIKE '%arthritis%',                         subject_id, NULL)) AS arthritis,
  COUNT(DISTINCT IF(title LIKE '%fracture%',                          subject_id, NULL)) AS fracture,
  COUNT(DISTINCT IF(title LIKE '%malignant%' AND title LIKE '%lung%', subject_id, NULL)) AS lung_cancer,
  COUNT(DISTINCT IF(title LIKE '%scoliosis%',                         subject_id, NULL)) AS scoliosis,
  COUNT(DISTINCT IF((title LIKE '%lung%' OR title LIKE '%pulmonary%')
                    AND title LIKE '%infection%',                     subject_id, NULL)) AS lung_infection
FROM dx
""").to_dataframe()
counts.T   # .T cho dễ đọc dạng cột

,0
pneumonia,26568
tuberculosis,2020
bronchitis,3683
arthritis,15160
fracture,32388
lung_cancer,9212
scoliosis,1594
lung_infection,1162


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"

confirmed = client.query(f"""
WITH dx AS (   -- bệnh nhân theo nhóm chẩn đoán nhiễm trùng
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
pos AS (       -- lần nằm viện có ít nhất 1 cấy dương tính
  SELECT DISTINCT subject_id, hadm_id
  FROM `{H}.microbiologyevents`
  WHERE org_name IS NOT NULL          -- có vi khuẩn mọc = dương tính
)
SELECT
  COUNT(DISTINCT IF(dx.title LIKE '%tuberculosis%', dx.subject_id, NULL)) AS tuberculosis_confirmed,
  COUNT(DISTINCT IF(dx.title LIKE '%pneumonia%',    dx.subject_id, NULL)) AS pneumonia_confirmed,
  COUNT(DISTINCT IF(dx.title LIKE '%bronchitis%',   dx.subject_id, NULL)) AS bronchitis_confirmed,
  COUNT(DISTINCT IF((dx.title LIKE '%lung%' OR dx.title LIKE '%pulmonary%')
                    AND dx.title LIKE '%infection%', dx.subject_id, NULL)) AS lung_infection_confirmed
FROM dx
JOIN pos USING (subject_id, hadm_id)   -- chỉ giữ lần nằm viện có cấy dương tính
""").to_dataframe()
confirmed.T

,0
tuberculosis_confirmed,526
pneumonia_confirmed,11145
bronchitis_confirmed,904
lung_infection_confirmed,428


**Có xét nghiệm vi sinh học**

In [ ]:
import pandas as pd
H = "physionet-data.mimiciv_3_1_hosp"

# --- (1) Số bệnh nhân được CHẨN ĐOÁN cho từng bệnh ---
diag = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
)
SELECT
  COUNT(DISTINCT IF(title LIKE '%pneumonia%',                          subject_id, NULL)) AS pneumonia,
  COUNT(DISTINCT IF(title LIKE '%tuberculosis%',                       subject_id, NULL)) AS tuberculosis,
  COUNT(DISTINCT IF(title LIKE '%bronchitis%',                         subject_id, NULL)) AS bronchitis,
  COUNT(DISTINCT IF(title LIKE '%arthritis%',                          subject_id, NULL)) AS arthritis,
  COUNT(DISTINCT IF(title LIKE '%fracture%',                           subject_id, NULL)) AS fracture,
  COUNT(DISTINCT IF(title LIKE '%malignant%' AND title LIKE '%lung%',  subject_id, NULL)) AS lung_cancer,
  COUNT(DISTINCT IF(title LIKE '%scoliosis%',                          subject_id, NULL)) AS scoliosis,
  COUNT(DISTINCT IF((title LIKE '%lung%' OR title LIKE '%pulmonary%')
                    AND title LIKE '%infection%',                      subject_id, NULL)) AS lung_infection
FROM dx
""").to_dataframe().T.reset_index()
diag.columns = ["disease", "n_diagnosed"]

In [ ]:
# --- (2) Số bệnh nhân có CẤY VI SINH DƯƠNG TÍNH (chỉ cho bệnh nhiễm trùng) ---
# Định nghĩa: vừa được chẩn đoán bệnh đó VÀ trong cùng lần nằm viện có org_name khác NULL.
# Riêng lao: yêu cầu đúng Mycobacterium.
micro = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
pos AS (   -- lần nằm viện có ít nhất 1 cấy dương tính (vi khuẩn mọc)
  SELECT DISTINCT subject_id, hadm_id FROM `{H}.microbiologyevents`
  WHERE org_name IS NOT NULL
),
tb_pos AS ( -- riêng lao: cấy dương tính với Mycobacterium
  SELECT DISTINCT subject_id, hadm_id FROM `{H}.microbiologyevents`
  WHERE org_name IS NOT NULL AND LOWER(org_name) LIKE '%mycobacterium%'
)
SELECT
  COUNT(DISTINCT IF(dx.title LIKE '%pneumonia%'  AND pos.subject_id IS NOT NULL, dx.subject_id, NULL)) AS pneumonia,
  COUNT(DISTINCT IF(dx.title LIKE '%tuberculosis%' AND tb.subject_id IS NOT NULL, dx.subject_id, NULL)) AS tuberculosis,
  COUNT(DISTINCT IF(dx.title LIKE '%bronchitis%' AND pos.subject_id IS NOT NULL, dx.subject_id, NULL)) AS bronchitis,
  COUNT(DISTINCT IF((dx.title LIKE '%lung%' OR dx.title LIKE '%pulmonary%')
        AND dx.title LIKE '%infection%' AND pos.subject_id IS NOT NULL, dx.subject_id, NULL)) AS lung_infection
FROM dx
LEFT JOIN pos    ON dx.subject_id = pos.subject_id AND dx.hadm_id = pos.hadm_id
LEFT JOIN tb_pos tb ON dx.subject_id = tb.subject_id  AND dx.hadm_id = tb.hadm_id
""").to_dataframe().T.reset_index()
micro.columns = ["disease", "n_micro_positive"]

In [ ]:
# --- (3) Gộp thành bảng so sánh sạch ---
applicable = {  # vi sinh có áp dụng được không + ghi chú
    "pneumonia":      ("Có (một phần)", "Cấy đờm/máu có thể (+); nhiều ca chẩn đoán bằng lâm sàng + X-quang"),
    "tuberculosis":   ("Có",           "Xác nhận bằng cấy Mycobacterium (chỉ bắt lao HOẠT ĐỘNG, không bắt lao tiềm ẩn)"),
    "bronchitis":     ("Hạn chế",      "Phần lớn do virus; cấy vi khuẩn ít khi xác nhận"),
    "lung_infection": ("Có (một phần)", "Tùy tác nhân; cấy có thể (+) hoặc (-)"),
    "arthritis":      ("Không",        "Bệnh khớp; chẩn đoán bằng lâm sàng/hình ảnh, không qua vi sinh"),
    "fracture":       ("Không",        "Chấn thương xương; chẩn đoán bằng X-quang/CT"),
    "lung_cancer":    ("Không",        "Ung thư; chẩn đoán bằng hình ảnh + giải phẫu bệnh"),
    "scoliosis":      ("Không",        "Biến dạng cột sống; chẩn đoán bằng hình ảnh"),
}

table = diag.merge(micro, on="disease", how="left")
table["micro_applicable"] = table["disease"].map(lambda d: applicable[d][0])
table["note"]             = table["disease"].map(lambda d: applicable[d][1])
# Bệnh không áp dụng vi sinh -> để trống thay vì 0 cho khỏi gây hiểu nhầm
table.loc[table["micro_applicable"] == "Không", "n_micro_positive"] = pd.NA

table = table.sort_values("n_diagnosed", ascending=False).reset_index(drop=True)
table

,disease,n_diagnosed,n_micro_positive,micro_applicable,note
0,fracture,32388,<NA>,Không,Chấn thương xương; chẩn đoán bằng X-quang/CT
1,pneumonia,26568,11145,Có (một phần),Cấy đờm/máu có thể (+); nhiều ca chẩn đoán bằn...
2,arthritis,15160,<NA>,Không,"Bệnh khớp; chẩn đoán bằng lâm sàng/hình ảnh, k..."
3,lung_cancer,9212,<NA>,Không,Ung thư; chẩn đoán bằng hình ảnh + giải phẫu bệnh
4,bronchitis,3683,904,Hạn chế,Phần lớn do virus; cấy vi khuẩn ít khi xác nhận
5,tuberculosis,2020,95,Có,Xác nhận bằng cấy Mycobacterium (chỉ bắt lao H...
6,scoliosis,1594,<NA>,Không,Biến dạng cột sống; chẩn đoán bằng hình ảnh
7,lung_infection,1162,428,Có (một phần),Tùy tác nhân; cấy có thể (+) hoặc (-)


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"
I = "physionet-data.mimiciv_3_1_icu"

cohort = client.query(f"""
  SELECT DISTINCT d.subject_id
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
  JOIN `{H}.microbiologyevents` m ON d.subject_id = m.subject_id AND d.hadm_id = m.hadm_id
  WHERE LOWER(dd.long_title) LIKE '%tuberculosis%'
    AND LOWER(m.org_name) LIKE '%mycobacterium%'
""").to_dataframe()

ids = ",".join(map(str, cohort["subject_id"].tolist()))
print("Số bệnh nhân:", len(cohort))

Số bệnh nhân: 95


In [ ]:
counts = client.query(f"""
SELECT
  (SELECT COUNT(*) FROM `{H}.labevents`         WHERE subject_id IN ({ids})) AS labevents_rows,
  (SELECT COUNT(*) FROM `{H}.diagnoses_icd`     WHERE subject_id IN ({ids})) AS diagnoses_rows,
  (SELECT COUNT(*) FROM `{H}.microbiologyevents`WHERE subject_id IN ({ids})) AS microbiology_rows,
  (SELECT COUNT(*) FROM `{H}.prescriptions`     WHERE subject_id IN ({ids})) AS prescriptions_rows,
  (SELECT COUNT(*) FROM `{H}.admissions`        WHERE subject_id IN ({ids})) AS admissions_rows,
  (SELECT COUNT(*) FROM `{I}.chartevents`       WHERE subject_id IN ({ids})) AS chartevents_rows,
  (SELECT COUNT(*) FROM `{I}.icustays`          WHERE subject_id IN ({ids})) AS icustays_rows
""").to_dataframe()
counts.T

,0
labevents_rows,215967
diagnoses_rows,5289
microbiology_rows,7879
prescriptions_rows,21266
admissions_rows,390
chartevents_rows,703563
icustays_rows,98


**Xem xét những xét nghiệm thường được chỉ định nhiều trong 2 bệnh lung_cancer và lung_infection để xem xét tính đặc hiệu, giúp phân loại âm/dương tính trong cohort được chẩn đoán bệnh ban đầu**

In [ ]:
relaxed = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
target AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE (title LIKE '%malignant%' AND title LIKE '%lung%')
     OR ((title LIKE '%lung%' OR title LIKE '%pulmonary%') AND title LIKE '%infection%')),
others AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE title LIKE '%pneumonia%' OR title LIKE '%tuberculosis%' OR title LIKE '%bronchitis%'
     OR title LIKE '%arthritis%' OR title LIKE '%fracture%' OR title LIKE '%scoliosis%'),
n_o AS (SELECT COUNT(*) c FROM others),
it AS (SELECT le.itemid, COUNT(DISTINCT le.hadm_id) adm_t
       FROM `{H}.labevents` le JOIN target USING(subject_id,hadm_id) GROUP BY le.itemid),
io AS (SELECT le.itemid, COUNT(DISTINCT le.hadm_id) adm_o
       FROM `{H}.labevents` le JOIN others USING(subject_id,hadm_id) GROUP BY le.itemid)
SELECT dl.label, it.adm_t, COALESCE(io.adm_o,0) AS adm_o,
       ROUND(COALESCE(io.adm_o,0)/(SELECT c FROM n_o),4) AS frac_in_others
FROM it LEFT JOIN io USING (itemid)
LEFT JOIN `{H}.d_labitems` dl USING (itemid)
WHERE it.adm_t >= 5 AND COALESCE(io.adm_o,0)/(SELECT c FROM n_o) < 0.01
ORDER BY it.adm_t DESC
""").to_dataframe()
relaxed

,label,adm_t,adm_o,frac_in_others
0,Carcinoembyronic Antigen (CEA),552,993,0.0095
1,"proBNP, Pleural",449,838,0.0080
2,Eosinophils,442,793,0.0076
3,Granulocyte Count,306,860,0.0082
4,"Triglycerides, Pleural",302,628,0.0060
...,...,...,...,...
367,NRBC,5,69,0.0007
368,CD20 Absolute Count,5,34,0.0003
369,CD20 %,5,34,0.0003
370,CD5 %,5,37,0.0004


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"
enrich = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
target AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE (title LIKE '%malignant%' AND title LIKE '%lung%')
     OR ((title LIKE '%lung%' OR title LIKE '%pulmonary%') AND title LIKE '%infection%')),
others AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE title LIKE '%pneumonia%' OR title LIKE '%tuberculosis%' OR title LIKE '%bronchitis%'
     OR title LIKE '%arthritis%' OR title LIKE '%fracture%' OR title LIKE '%scoliosis%'),
cnt_t AS (SELECT COUNT(*) c FROM target),
cnt_o AS (SELECT COUNT(*) c FROM others),
it AS (SELECT le.itemid, COUNT(DISTINCT le.hadm_id) adm_t
       FROM `{H}.labevents` le JOIN target USING(subject_id,hadm_id) GROUP BY le.itemid),
io AS (SELECT le.itemid, COUNT(DISTINCT le.hadm_id) adm_o
       FROM `{H}.labevents` le JOIN others USING(subject_id,hadm_id) GROUP BY le.itemid)
SELECT dl.label,
       it.adm_t, COALESCE(io.adm_o,0) AS adm_o,
       ROUND(it.adm_t/(SELECT c FROM cnt_t),4)                 AS frac_target,
       ROUND(COALESCE(io.adm_o,0)/(SELECT c FROM cnt_o),4)     AS frac_others,
       ROUND( (it.adm_t/(SELECT c FROM cnt_t)) /
              NULLIF(COALESCE(io.adm_o,0)/(SELECT c FROM cnt_o),0), 2) AS enrichment
FROM it LEFT JOIN io USING (itemid)
LEFT JOIN `{H}.d_labitems` dl USING (itemid)
WHERE it.adm_t >= 20
ORDER BY enrichment DESC
LIMIT 30
""").to_dataframe()
enrich

,label,adm_t,adm_o,frac_target,frac_others,enrichment
0,"Miscellaneous, CSF",52,16,0.0024,0.0002,15.98
1,Cancer Antigen 27.29,76,95,0.0036,0.0009,3.93
2,Other,785,1234,0.0369,0.0118,3.13
3,CA-125,155,247,0.0073,0.0024,3.09
4,Eosinophils,442,793,0.0208,0.0076,2.74
5,Carcinoembyronic Antigen (CEA),552,993,0.0260,0.0095,2.73
6,"LD, Body Fluid",103,192,0.0048,0.0018,2.64
7,"Albumin, Body Fluid",102,190,0.0048,0.0018,2.64
8,"proBNP, Pleural",449,838,0.0211,0.0080,2.64
9,"Hematocrit, Pleural",201,387,0.0095,0.0037,2.55


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"

abn = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
target AS (
  SELECT DISTINCT subject_id, hadm_id FROM dx
  WHERE (title LIKE '%malignant%' AND title LIKE '%lung%')
     OR ((title LIKE '%lung%' OR title LIKE '%pulmonary%') AND title LIKE '%infection%')
)
SELECT dl.label, dl.category,
       COUNT(*)                              AS n_results,
       COUNTIF(le.flag IS NOT NULL)          AS n_abnormal,
       ROUND(100*COUNTIF(le.flag IS NOT NULL)/COUNT(*),1) AS pct_abnormal
FROM `{H}.labevents` le
JOIN target USING (subject_id, hadm_id)
LEFT JOIN `{H}.d_labitems` dl USING (itemid)
WHERE dl.label IS NOT NULL
  AND LOWER(dl.label) NOT IN ('other','miscellaneous','unknown')
GROUP BY dl.label, dl.category
HAVING n_results >= 100          -- đủ mẫu để tỷ lệ đáng tin
ORDER BY pct_abnormal DESC, n_abnormal DESC
LIMIT 30
""").to_dataframe()
abn

,label,category,n_results,n_abnormal,pct_abnormal
0,"Total Nucleated Cells, Ascites",Hematology,407,405,99.5
1,"Total Nucleated Cells, Pleural",Hematology,1638,1627,99.3
2,"RBC, Pleural",Hematology,1506,1496,99.3
3,Macrophages,Hematology,1184,1175,99.2
4,"RBC, Ascites",Hematology,392,389,99.2
5,Mesothelial Cell,Hematology,244,242,99.2
6,Mesothelial Cells,Hematology,845,836,98.9
7,Nucleated Red Cells,Hematology,4106,4038,98.3
8,Hyaline Casts,Hematology,3165,3093,97.7
9,Blasts,Hematology,843,824,97.7


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"
abn_enrich = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
target AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE (title LIKE '%malignant%' AND title LIKE '%lung%')
     OR ((title LIKE '%lung%' OR title LIKE '%pulmonary%') AND title LIKE '%infection%')),
others AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE title LIKE '%pneumonia%' OR title LIKE '%tuberculosis%' OR title LIKE '%bronchitis%'
     OR title LIKE '%arthritis%' OR title LIKE '%fracture%' OR title LIKE '%scoliosis%'),
t AS (SELECT le.itemid,
        COUNT(DISTINCT le.subject_id) n_pat,
        COUNT(*) n,
        COUNTIF(le.flag IS NOT NULL) ab
      FROM `{H}.labevents` le JOIN target USING(subject_id,hadm_id) GROUP BY le.itemid),
o AS (SELECT le.itemid, COUNT(*) n, COUNTIF(le.flag IS NOT NULL) ab
      FROM `{H}.labevents` le JOIN others USING(subject_id,hadm_id) GROUP BY le.itemid)
SELECT dl.label, dl.category,
       t.n_pat AS n_patients_target,
       t.n AS n_target, ROUND(100*t.ab/t.n,1) AS pct_abn_target,
       ROUND(100*o.ab/NULLIF(o.n,0),1)        AS pct_abn_others,
       ROUND( (t.ab/t.n) / NULLIF(o.ab/NULLIF(o.n,0),0), 2) AS abn_ratio

FROM t JOIN o USING (itemid)
LEFT JOIN `{H}.d_labitems` dl USING (itemid)
WHERE t.n >= 500 AND o.n >= 500
  AND t.n_pat >=30
  AND dl.label IS NOT NULL
  AND LENGTH(dl.label) >= 3                       -- bỏ label 1-2 ký tự như "H"
  AND LOWER(dl.label) NOT IN ('other','miscellaneous','unknown','h')
ORDER BY abn_ratio DESC
LIMIT 30
""").to_dataframe()
abn_enrich

,label,category,n_patients_target,n_target,pct_abn_target,pct_abn_others,abn_ratio
0,CK-MB Index,Chemistry,387,766,50.3,32.1,1.57
1,Hepatitis B Surface Antigen,Chemistry,772,865,1.0,0.7,1.48
2,Ketone,Hematology,5498,12838,11.1,7.6,1.46
3,tacroFK,Chemistry,98,1084,43.9,30.5,1.44
4,Spherocytes,Hematology,413,759,46.2,36.8,1.26
5,Bands,Hematology,2222,8622,16.5,13.3,1.24
6,Hepatitis B Virus Core Antibody,Chemistry,671,757,12.8,10.4,1.23
7,"Potassium, Whole Blood",Blood Gas,2507,7889,21.9,18.5,1.18
8,Haptoglobin,Chemistry,1523,2330,61.4,53.1,1.16
9,Carcinoembyronic Antigen (CEA),Chemistry,531,572,59.4,51.1,1.16


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"

value_cmp = client.query(f"""
WITH dx AS (
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
target AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE (title LIKE '%malignant%' AND title LIKE '%lung%')
     OR ((title LIKE '%lung%' OR title LIKE '%pulmonary%') AND title LIKE '%infection%')),
others AS (SELECT DISTINCT subject_id,hadm_id FROM dx
  WHERE title LIKE '%pneumonia%' OR title LIKE '%tuberculosis%' OR title LIKE '%bronchitis%'
     OR title LIKE '%arthritis%' OR title LIKE '%fracture%' OR title LIKE '%scoliosis%'),
t AS (
  SELECT le.itemid,
         COUNT(DISTINCT le.subject_id)                          AS n_pat_t,
         COUNT(le.valuenum)                                     AS n_t,
         APPROX_QUANTILES(le.valuenum, 2)[OFFSET(1)]            AS median_t
  FROM `{H}.labevents` le JOIN target USING(subject_id,hadm_id)
  WHERE le.valuenum IS NOT NULL
  GROUP BY le.itemid
),
o AS (
  SELECT le.itemid,
         COUNT(le.valuenum)                                     AS n_o,
         APPROX_QUANTILES(le.valuenum, 2)[OFFSET(1)]            AS median_o
  FROM `{H}.labevents` le JOIN others USING(subject_id,hadm_id)
  WHERE le.valuenum IS NOT NULL
  GROUP BY le.itemid
)
SELECT dl.label, dl.category, dl.fluid,
       t.n_pat_t, t.n_t, o.n_o,
       ROUND(t.median_t, 2) AS median_target,
       ROUND(o.median_o, 2) AS median_others,
       ROUND( SAFE_DIVIDE(t.median_t, NULLIF(o.median_o,0)), 2) AS median_ratio
FROM t JOIN o USING (itemid)
LEFT JOIN `{H}.d_labitems` dl USING (itemid)
WHERE t.n_t >= 500 AND o.n_o >= 500
  AND t.n_pat_t >= 30
  AND dl.label IS NOT NULL
  AND LENGTH(dl.label) >= 3
  AND LOWER(dl.label) NOT IN ('other','miscellaneous','unknown','h')
  AND o.median_o IS NOT NULL AND o.median_o <> 0
ORDER BY median_ratio DESC
LIMIT 40
""").to_dataframe()
value_cmp

,label,category,fluid,n_pat_t,n_t,n_o,median_target,median_others,median_ratio
0,Carcinoembyronic Antigen (CEA),Chemistry,Blood,487,519,989,11.30,4.60,2.46
1,Ketone,Hematology,Urine,1216,2030,8498,20.00,10.00,2.00
2,CK-MB Index,Chemistry,Blood,387,766,6872,6.20,3.40,1.82
3,Monocytes,Hematology,Cerebrospinal Fluid,324,583,3150,14.00,9.00,1.56
4,Granulocyte Count,Hematology,Blood,266,1024,6136,690.00,462.00,1.49
5,Glucose,Hematology,Urine,661,1008,6201,200.00,150.00,1.33
6,Lymphocytes,Hematology,Pleural,1118,1628,3621,33.00,26.00,1.27
7,"Lactate Dehydrogenase, Pleural",Chemistry,Pleural,1130,1611,3555,269.00,211.00,1.27
8,"Cholesterol, Pleural",Chemistry,Pleural,811,1111,2334,53.00,42.00,1.26
9,Haptoglobin,Chemistry,Blood,1449,2110,11886,230.00,185.00,1.24


In [ ]:
import pandas as pd
H = "physionet-data.mimiciv_3_1_hosp"

summary = client.query(f"""
WITH dx AS (   -- bệnh nhân + lần nằm viện theo nhóm chẩn đoán
  SELECT DISTINCT d.subject_id, d.hadm_id, LOWER(dd.long_title) AS title
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
micro AS (     -- trạng thái cấy vi sinh theo từng lần nằm viện
  SELECT subject_id, hadm_id,
         MAX(IF(org_name IS NOT NULL, 1, 0)) AS has_pos,   -- có ≥1 cấy dương
         COUNT(*)                             AS n_cultures  -- có làm cấy hay không
  FROM `{H}.microbiologyevents`
  GROUP BY subject_id, hadm_id
),
labeled AS (   -- gán nhãn từng bệnh cho mỗi (subject_id, hadm_id) + trạng thái cấy
  SELECT dx.subject_id, dx.hadm_id,
         CASE
           WHEN dx.title LIKE '%tuberculosis%' THEN 'tuberculosis'
           WHEN dx.title LIKE '%pneumonia%'    THEN 'pneumonia'
           WHEN dx.title LIKE '%bronchitis%'   THEN 'bronchitis'
           WHEN (dx.title LIKE '%lung%' OR dx.title LIKE '%pulmonary%')
                 AND dx.title LIKE '%infection%' THEN 'lung_infection'
         END AS disease,
         IF(m.subject_id IS NULL, 0, 1) AS has_culture,   -- 1 = có làm cấy
         COALESCE(m.has_pos, 0)          AS has_pos
  FROM dx
  LEFT JOIN micro m USING (subject_id, hadm_id)
)
SELECT
  disease,
  COUNT(DISTINCT subject_id)                                          AS n_diagnosed,
  COUNT(DISTINCT IF(has_culture = 1, subject_id, NULL))               AS n_with_culture,
  COUNT(DISTINCT IF(has_culture = 0, subject_id, NULL))               AS n_no_culture,
  COUNT(DISTINCT IF(has_pos = 1, subject_id, NULL))                   AS n_positive,
  COUNT(DISTINCT IF(has_culture = 1 AND has_pos = 0, subject_id, NULL)) AS n_negative
FROM labeled
WHERE disease IS NOT NULL
GROUP BY disease
ORDER BY n_diagnosed DESC
""").to_dataframe()

summary

,disease,n_diagnosed,n_with_culture,n_no_culture,n_positive,n_negative
0,pneumonia,26568,22056,6522,11145,12861
1,bronchitis,3683,2388,1707,904,1653
2,tuberculosis,2020,1357,944,526,981
3,lung_infection,1162,962,266,428,581


In [ ]:
tbl = summary.copy()

# Tỉ lệ khuyết thiếu xét nghiệm = số ca KHÔNG làm cấy / tổng số ca được chẩn đoán
tbl["missing_rate_%"]   = (100 * tbl["n_no_culture"]  / tbl["n_diagnosed"]).round(1)

# Trong nhóm CÓ làm cấy: tỉ lệ dương và âm
tbl["positive_rate_%"]  = (100 * tbl["n_positive"] / tbl["n_with_culture"]).round(1)
tbl["negative_rate_%"]  = (100 * tbl["n_negative"] / tbl["n_with_culture"]).round(1)

# Tỉ lệ dương / âm
tbl["pos_neg_ratio"]    = (tbl["n_positive"] / tbl["n_negative"].replace(0, pd.NA)).round(2)

cols = ["disease", "n_diagnosed", "n_with_culture", "n_no_culture",
        "n_positive", "n_negative",
        "missing_rate_%", "positive_rate_%", "negative_rate_%", "pos_neg_ratio"]
tbl[cols]

,disease,n_diagnosed,n_with_culture,n_no_culture,n_positive,n_negative,missing_rate_%,positive_rate_%,negative_rate_%,pos_neg_ratio
0,pneumonia,26568,22056,6522,11145,12861,24.5,50.5,58.3,0.87
1,bronchitis,3683,2388,1707,904,1653,46.3,37.9,69.2,0.55
2,tuberculosis,2020,1357,944,526,981,46.7,38.8,72.3,0.54
3,lung_infection,1162,962,266,428,581,22.9,44.5,60.4,0.74


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"
PID = 10032409  # đổi thành subject_id bạn quan tâm

micro_detail = client.query(f"""
SELECT
  subject_id,
  hadm_id,
  charttime,
  micro_specimen_id,
  spec_type_desc            AS specimen,      -- loại bệnh phẩm
  test_name,                                  -- xét nghiệm được chỉ định
  org_name,                                   -- vi khuẩn định danh (NULL = không mọc)
  IF(org_name IS NULL, 'no_growth/negative', 'positive') AS culture_result,
  ab_name,                                    -- kháng sinh (nếu có kháng sinh đồ)
  interpretation                              -- S/I/R
FROM `{H}.microbiologyevents`
WHERE subject_id = {PID}
ORDER BY hadm_id, charttime, micro_specimen_id, org_name
""").to_dataframe()

micro_detail.head(100)

,subject_id,hadm_id,charttime,micro_specimen_id,specimen,test_name,org_name,culture_result,ab_name,interpretation
0,10032409,<NA>,NaT,277719,URINE,URINE CULTURE,ENTEROCOCCUS SP.,positive,None,None
1,10032409,<NA>,NaT,4063504,URINE,Legionella Urinary Antigen,None,no_growth/negative,None,None
2,10032409,<NA>,NaT,5763979,URINE,URINE CULTURE,None,no_growth/negative,None,None
3,10032409,<NA>,2127-03-14 20:30:00,4392488,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None
4,10032409,<NA>,2128-06-22 13:35:00,2186770,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None
5,10032409,<NA>,2128-06-22 14:15:00,7534438,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None
6,10032409,<NA>,2128-10-11 19:53:00,7610092,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None
7,10032409,<NA>,2129-11-26 11:30:00,7337515,URINE,URINE CULTURE,GRAM POSITIVE BACTERIA,positive,None,None
8,10032409,<NA>,2130-01-12 15:30:00,2182529,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None
9,10032409,<NA>,2130-01-12 17:40:00,9825451,BLOOD CULTURE,"Blood Culture, Routine",None,no_growth/negative,None,None


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"
PID = 16853337   # đổi thành subject_id bạn quan tâm

micro_per_visit = client.query(f"""
WITH per_spec AS (   -- gộp về mức từng mẫu cấy
  SELECT subject_id, hadm_id, micro_specimen_id,
         MAX(IF(org_name IS NOT NULL, 1, 0)) AS spec_positive
  FROM `{H}.microbiologyevents`
  WHERE subject_id = {PID}
  GROUP BY subject_id, hadm_id, micro_specimen_id
)
SELECT
  m.subject_id,
  m.hadm_id,
  COUNT(DISTINCT m.micro_specimen_id)                              AS n_specimens,      -- số mẫu cấy trong lần khám
  COUNT(DISTINCT IF(s.spec_positive = 1, m.micro_specimen_id, NULL)) AS n_pos_specimens, -- số mẫu dương
  MAX(IF(m.org_name IS NOT NULL, 1, 0))                            AS has_positive,     -- lần khám có ≥1 mẫu dương?
  STRING_AGG(DISTINCT m.org_name, ' | ')                           AS organisms         -- danh sách vi khuẩn định danh
FROM `{H}.microbiologyevents` m
JOIN per_spec s USING (subject_id, hadm_id, micro_specimen_id)
WHERE m.subject_id = {PID}
GROUP BY m.subject_id, m.hadm_id
ORDER BY m.hadm_id
""").to_dataframe()

micro_per_visit

,subject_id,hadm_id,n_specimens,n_pos_specimens,has_positive,organisms
0,16853337,29712263,2,0,0,None


In [ ]:
H = "physionet-data.mimiciv_3_1_hosp"

cohort_micro = client.query(f"""
WITH dx AS (   -- gán nhãn bệnh cho mỗi lần nằm viện
  SELECT DISTINCT d.subject_id, d.hadm_id,
         CASE
           WHEN LOWER(dd.long_title) LIKE '%tuberculosis%' THEN 'tuberculosis'
           WHEN LOWER(dd.long_title) LIKE '%pneumonia%'    THEN 'pneumonia'
           WHEN LOWER(dd.long_title) LIKE '%bronchitis%'   THEN 'bronchitis'
           WHEN (LOWER(dd.long_title) LIKE '%lung%' OR LOWER(dd.long_title) LIKE '%pulmonary%')
                 AND LOWER(dd.long_title) LIKE '%infection%' THEN 'lung_infection'
         END AS disease
  FROM `{H}.diagnoses_icd` d
  JOIN `{H}.d_icd_diagnoses` dd USING (icd_code, icd_version)
),
micro AS (     -- tóm tắt cấy theo lần nằm viện
  SELECT subject_id, hadm_id,
         COUNT(DISTINCT micro_specimen_id)          AS n_specimens,
         MAX(IF(org_name IS NOT NULL, 1, 0))        AS has_positive,
         STRING_AGG(DISTINCT org_name, ' | ')       AS organisms
  FROM `{H}.microbiologyevents`
  GROUP BY subject_id, hadm_id
)
SELECT
  dx.disease,
  dx.subject_id,
  dx.hadm_id,
  IF(m.hadm_id IS NULL, 0, 1)              AS has_culture,   -- lần khám có làm cấy không
  COALESCE(m.n_specimens, 0)               AS n_specimens,
  COALESCE(m.has_positive, 0)              AS has_positive,   -- có định danh được vi khuẩn không
  m.organisms                              AS organisms       -- vi khuẩn định danh (nếu có)
FROM dx
LEFT JOIN micro m USING (subject_id, hadm_id)
WHERE dx.disease IS NOT NULL
ORDER BY dx.disease, dx.subject_id, dx.hadm_id
""").to_dataframe()

cohort_micro.head(100)

,disease,subject_id,hadm_id,has_culture,n_specimens,has_positive,organisms
0,bronchitis,10001884,23594368,0,0,0,None
1,bronchitis,10002155,20345487,1,3,0,None
2,bronchitis,10011938,23501236,0,0,0,None
3,bronchitis,10011938,29339128,1,2,0,None
4,bronchitis,10014354,29600294,1,4,1,YEAST
...,...,...,...,...,...,...,...
95,bronchitis,10212427,28481664,0,0,0,None
96,bronchitis,10213275,21902946,1,7,1,YEAST
97,bronchitis,10213275,24227559,1,16,1,STAPH AUREUS COAG + | ENTEROCOCCUS FAECALIS
98,bronchitis,10215793,25471024,1,1,0,None
